# OpenShell Sandbox — `OpenShellSandbox`

This notebook walks through using
[`langchain-nvidia-openshell`](https://github.com/langchain-ai/langchain-nvidia/tree/main/libs/openshell)
to plug an **NVIDIA OpenShell** sandbox into a LangChain Deep Agents
workflow as a remote tool runtime — the classic *sandbox-as-tool* pattern
described in the
[LangChain sandbox docs](https://docs.langchain.com/oss/python/deepagents/sandboxes).

Concretely:

* Your agent (driven by `ChatNVIDIA`) runs on the host.
* Tool calls — shell commands, file uploads, file downloads — are dispatched
  into a policy-enforced OpenShell sandbox over gRPC.
* OpenShell handles the isolation: filesystem (Landlock), syscalls (seccomp),
  and outbound network (HTTP CONNECT proxy + OPA/Rego policy).

> ⚠️ **Alpha**. NVIDIA OpenShell is alpha software. The API surface is
> stable enough for experimentation but expect breakage as it matures.


## 1. Prerequisites

You need three things on your machine:

1. **The OpenShell CLI / gateway**. Install with:

   ```bash
   uv tool install -U openshell
   # or
   curl -LsSf https://raw.githubusercontent.com/NVIDIA/OpenShell/main/install.sh | sh
   ```

2. **A local sandbox + gateway**. The first `openshell sandbox create` call
   also boots a gateway on `127.0.0.1:8080` and writes mTLS material to
   `~/.config/openshell/`:

   ```bash
   openshell sandbox create --keep --no-tty -- bash
   ```

   Leave the sandbox running in the background — we'll connect to it from
   Python.

3. **Python packages**:

   ```bash
   pip install --upgrade --quiet \
       langchain-nvidia-ai-endpoints \
       langchain-nvidia-openshell \
       deepagents \
       openshell
   ```

Set your NVIDIA API key (any service account that can hit your chosen
ChatNVIDIA model):


In [ ]:
import getpass
import os

if not os.environ.get("NVIDIA_API_KEY"):
    os.environ["NVIDIA_API_KEY"] = getpass.getpass("NVIDIA_API_KEY: ")


## 2. Construct an `OpenShellSandbox`

`OpenShellSandbox` is *lifecycle-agnostic*: you bring an already-entered
`openshell.Sandbox` (or a `SandboxSession`) and we wrap it. This matches the
convention used by the Daytona, Modal, and Runloop integrations and means
you can attach to a long-lived sandbox just as easily as you can spin one up
ephemerally.


In [ ]:
import openshell
from langchain_nvidia_openshell import OpenShellSandbox

# Enter the OpenShell context manually so we can keep it alive across cells
sandbox = openshell.Sandbox()
sandbox.__enter__()

backend = OpenShellSandbox(sandbox=sandbox)
print("sandbox id:", backend.id)


## 3. Run a shell command

`backend.execute(command)` returns an
[`ExecuteResponse`](https://github.com/langchain-ai/deepagents/blob/main/libs/deepagents/deepagents/backends/protocol.py)
whose `output` is the **combined** stdout + stderr — exactly what an LLM
expects when reading tool results.


In [ ]:
result = backend.execute("uname -a && python3 --version")
print("exit:", result.exit_code)
print("truncated:", result.truncated)
print("---")
print(result.output)


### Per-call timeouts

`execute(command, timeout=...)` enforces a per-call deadline. Setting
`timeout=0` means "no timeout"; the default is the value passed to the
constructor (1800s).


In [ ]:
slow = backend.execute("sleep 5 && echo done", timeout=2)
print("exit:", slow.exit_code)   # 124 if a timeout was raised by the SDK
print("output:", slow.output[:200])


### Custom shell

If your sandbox image lacks `bash` (e.g. busybox `sh`-only), override the
shell prefix:

```python
busybox_backend = OpenShellSandbox(sandbox=sandbox, shell=("sh", "-c"))
```


## 4. Upload and download files

`OpenShell` does not expose a typed file-upload SDK call, so we tunnel files
through `exec` itself: a tiny inline Python bootstrap reads base64 from
stdin (upload) or writes base64 to stdout (download). For the user this
remains the standard `BaseSandbox` interface.


In [ ]:
script = b"""
import sys
print('hello from inside the sandbox')
print('argv:', sys.argv[1:])
""".lstrip()

[upload] = backend.upload_files([("/sandbox/hello.py", script)])
print("upload error:", upload.error)

run = backend.execute("python3 /sandbox/hello.py first second")
print(run.output)

[download] = backend.download_files(["/sandbox/hello.py"])
print("download error:", download.error)
print("first 32 bytes:", download.content[:32])
assert download.content == script, "round-trip mismatch"


### Partial-success semantics

Every entry in the `files`/`paths` list gets its own response, so you can
batch uploads/downloads and discover which ones failed without re-running
the whole thing:


In [ ]:
mixed = backend.upload_files([
    ("relative/path.txt", b"this will fail"),     # invalid_path
    ("/sandbox/ok.txt", b"this will succeed"),
])
for r in mixed:
    print(r.path, "->", r.error)


## 5. Plug into a Deep Agent driven by `ChatNVIDIA`

The whole point of `BaseSandbox` is that it slots into
`deepagents.create_deep_agent(...)` as the `backend=` argument. The agent
then transparently uses our sandbox for `execute`/`read_file`/`write_file`/
`ls`/`grep`/etc. — every file op is built on top of `execute` by
`BaseSandbox`.


In [ ]:
from deepagents import create_deep_agent
from langchain_nvidia_ai_endpoints import ChatNVIDIA

agent = create_deep_agent(
    model=ChatNVIDIA(model="meta/llama-3.3-70b-instruct"),
    system_prompt=(
        "You are a careful coding agent. When you need to run code or "
        "inspect files, use the sandbox tools. Always print the result of "
        "any computation."
    ),
    backend=backend,
)

response = agent.invoke({
    "messages": [{
        "role": "user",
        "content": (
            "Inside the sandbox, write a Python script at /sandbox/sum.py "
            "that prints the sum of 1..100, run it, and report the output."
        ),
    }]
})

# Print just the assistant's last reply for brevity
print(response["messages"][-1].content)


## 6. Inspect the request the wrapper sends to OpenShell

The wrapper translates `command: str` → `[bash, -c, command]` argv before
issuing the gRPC `ExecSandbox` call. You can confirm this by intercepting
the underlying `Sandbox.exec`:


In [ ]:
original_exec = sandbox.exec
recorded = []

def spy(command, **kwargs):
    recorded.append({"command": list(command), "kwargs": kwargs})
    return original_exec(command, **kwargs)

sandbox.exec = spy
try:
    backend.execute("echo wrapped")
finally:
    sandbox.exec = original_exec

print(recorded[0]["command"])      # ['bash', '-c', 'echo wrapped']
print({k: v for k, v in recorded[0]["kwargs"].items() if k != "stdin"})


## 7. Cleanup

Always close the OpenShell sandbox when you're done — otherwise it stays
running on your gateway. The standard idiom is `with openshell.Sandbox() as
sb: ...`. Since we used `__enter__` manually here, we need to call
`__exit__`:


In [ ]:
sandbox.__exit__(None, None, None)
print("sandbox cleaned up")


## How `OpenShellSandbox` works

```text
                ┌─────────────────────────────────────┐
                │ Host                                │
                │                                     │
                │  ChatNVIDIA(...)                    │
                │     │                               │
                │     ▼                               │
                │  create_deep_agent(                 │
                │    model=...,                       │
                │    backend=OpenShellSandbox(sb)     │
                │  )                                  │
                │     │                               │
                │     ▼                               │
                │  BaseSandbox.execute / upload /     │
                │  download / read / write / ...      │
                │     │                               │
                │     ▼                               │
                │  openshell.Sandbox.exec(            │
                │    ['bash','-c','...']              │
                │  )                                  │
                └────────────────┬────────────────────┘
                                 │ gRPC over mTLS
                                 ▼
                ┌─────────────────────────────────────┐
                │ OpenShell Gateway (127.0.0.1:8080)  │
                └────────────────┬────────────────────┘
                                 │ relay-tunneled SSH
                                 ▼
                ┌─────────────────────────────────────┐
                │ Sandbox (Docker / VM / K8s)         │
                │   • Landlock filesystem             │
                │   • seccomp syscall filter          │
                │   • OPA-policy egress proxy         │
                │   • Python 3.13, Node 22, gh, git   │
                └─────────────────────────────────────┘
```

| Method | What we do |
|---|---|
| `execute(cmd)` | `Sandbox.exec(['bash','-c', cmd])` → combine stdout+stderr |
| `upload_files` | `Sandbox.exec(['python3','-c', _UPLOAD_BOOTSTRAP], stdin=b64(content), env={...path})` |
| `download_files` | `Sandbox.exec(['python3','-c', _DOWNLOAD_BOOTSTRAP], env={...path})` → b64-decode stdout |
| `id` | `Sandbox.id` (the gRPC sandbox id) |
| `aexecute`, `read`, `write`, `edit`, `ls`, `glob`, `grep`, … | inherited from `BaseSandbox` |

### Comparison with other LangChain sandbox backends

| Backend | Wire | Constructor argument | File transfer |
|---|---|---|---|
| `DaytonaSandbox` | Daytona REST SDK | `sandbox=daytona.Daytona().create()` | `fs.upload_files` / `fs.download_files` |
| `ModalSandbox` | Modal serverless SDK | `sandbox=modal.Sandbox.create(app=...)` | uses Modal volumes |
| `RunloopSandbox` | Runloop REST SDK | `devbox=client.devbox.create()` | `devbox.file.upload/download` |
| **`OpenShellSandbox`** | **NVIDIA OpenShell gRPC** | **`sandbox=openshell.Sandbox()`** | **`exec`-tunneled base64** |
| `LangSmithSandbox` | LangSmith SDK | `sandbox=client.create_sandbox(...)` | LangSmith API |


## Related links

* [NVIDIA OpenShell](https://github.com/NVIDIA/OpenShell) — the runtime.
* [OpenShell docs](https://docs.nvidia.com/openshell/latest/) — install,
  policy schema, observability.
* [LangChain Deep Agents — sandboxes](https://docs.langchain.com/oss/python/deepagents/sandboxes)
  — the abstraction we conform to.
* [`langchain-nvidia-openshell`](https://github.com/langchain-ai/langchain-nvidia/tree/main/libs/openshell)
  — this package's source and tests.
* [`ChatNVIDIA`](https://docs.nvidia.com/ai-endpoints) — the LLM driving the
  agent.
